# Differential gene expression

In [ ]:
# eval "$(conda shell.bash hook)"
# conda init
# conda activate /work/islet_cartography_scrna/scrna_cartography_dreampy
# python -m ipykernel install --user --name scrna_cartography_dreampy --display-name "dreampy"

In [1]:
# Path and system utilities
import os                    # Operating system interface
import sys                   # System-specific parameters and functions
import glob                  # File pattern matching
from pathlib import Path     # Object-oriented filesystem paths
from pyhere import here      # Reproducible project paths
import gc

# Single-cell data handling
import anndata as ad            # Core data structure for single-cell data
import scanpy as sc

# dream
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats
import dreampy as dp

# Parallel processing
from joblib import Parallel, delayed, parallel_backend

# dataframes
import pandas as pd
import numpy as np
from collections import defaultdict

# Custom modules and functions
sys.path.append(str(here('scripts/misc')))  # Add custom script path to system
import misc as mi
import diff_genes as dg

In [2]:
# Paths
base_dir = str(here('data/annotate/'))
plot_dir = os.path.join(base_dir, 'plot') 
files_dir = os.path.join(base_dir, 'files') 
diffg_dir = os.path.join(base_dir, 'deseq_onevsother') 
tmp_dir = os.path.join(base_dir, 'tmp') 

anndata_dir = str(here('data/anndata/'))

mi.create_directories(os.path.join(base_dir, 'deseq_onevsother'))
mi.create_directories(os.path.join(base_dir, 'tmp'))

/work/islet_cartography_scrna/data/annotate/deseq_onevsother Directory already exists!
/work/islet_cartography_scrna/data/annotate/tmp Directory already exists!


In [3]:
adata = ad.read_h5ad(os.path.join(anndata_dir, "AH_combined.h5ad"))

## Differential gene expression

In [4]:
# Setup -----------------------------------------------------------------------------
anno_key   = "manual_annotation"
sample_key = "ic_id_platform_adjusted_sample"
donor_key  = "ic_id_donor_overall"
dataset_key  = "ic_id_dataset"
disease_key = "disease_harmonized"
target_celltypes = adata.obs[anno_key].unique()
inference = DefaultInference(n_cpus=-1)

#### Using all datasets - adjusting for sample

In [ ]:
all_results = []

# Loop over all cell types -----------------------------------------------------------------------------
for cluster_id in target_celltypes:

    print(f"\n==============================")
    print(f"Running: {cluster_id} vs other")
    print(f"==============================")

    comp = f"{cluster_id}_vs_other"
    adata.obs[comp] = (
        adata
        .obs[anno_key]
        .apply(lambda x: cluster_id if x == cluster_id else 'other'))
    
    adata.obs['assay'] = 'my_assay'
    
    # Pseudobulk aggregation (by comparison group + sample)
    pb = dp.aggregate_pseudobulk(
        adata,
        layer='counts',
        groupby=['assay', sample_key, comp]
    )

    try:
        min_cells = 50
        pb = dp.filter_samples(pb, min_cells=min_cells, min_samples=3)
        
        if pb.obs.groupby(comp)['assay'].count()[cluster_id] < 3:
            pb = dp.aggregate_pseudobulk(
            adata,
            layer='counts',
            groupby=['assay', sample_key, comp])

            min_cells = 10
            pb = dp.filter_samples(pb, min_cells=min_cells, min_samples=3)
        
        # Count matrix 
        counts_df = pd.DataFrame(
            pb.X.toarray(),
            columns=pb.var_names,
            index=pb.obs_names)

        # Meta data
        metadata_df = pb.obs[[sample_key, comp]].copy()
        metadata_df = metadata_df.set_index(counts_df.index)
        
        assert counts_df.index.equals(metadata_df.index), "Index are not equal!"
        
        formula = "~ {} + {}".format(sample_key, comp)
        
        dds = DeseqDataSet(
            counts=counts_df,
            metadata=metadata_df,
            design= formula,
            inference=inference
        )
        
        dds.deseq2()
        
        ds = DeseqStats(
            dds,
            contrast=(comp, cluster_id, 'other'), 
            inference = inference,
            quiet = True)
        
        # run wald test
        ds.run_wald_test()
        ds.summary()
        
        results = ds.results_df.copy()
        n_donors = ds.dds.shape[0]
        results['comparison'] = f"{cluster_id}_other"
        results['manual_annotation'] = cluster_id
        results['n_donors'] = n_donors
        results['min_cells'] = min_cells
    
        all_results.append(results)
        
    except Exception as e:
        print("Skipping:", cluster_id, e)

# Combine all results -------------------------------------------------------------------------
marker_results = pd.concat(all_results)
marker_results.to_csv(os.path.join(diffg_dir, f"deseq2_one_vs_all.csv"), index=True, index_label = "gene_symbol")

#### Per dataset - adjusting for donor

In [ ]:
# Calculate percent of genes expressed
perc_all = dg.compute_pct_expressing(adata, anno_key)
perc_all.to_csv(os.path.join(diffg_dir, f"percentage_of_genes_expressed_manual_annotation.csv"), index=True, index_label = "gene_symbol")

# per celltype vs other
for cluster_id in target_celltypes:

    meta_results = []

    print(f"\n==============================")
    print(f"Running: {cluster_id} vs other")
    print(f"==============================")

    comp = f"{cluster_id}_vs_other"
    
    adata.obs[comp] = (
        adata
        .obs[anno_key]
        .apply(lambda x: cluster_id if x == cluster_id else 'other'))

    # save
    perc_comp = dg.compute_pct_expressing(adata, comp)
    perc_comp.to_csv(os.path.join(diffg_dir, f"percentage_of_genes_expressed_{comp}.csv"), index=True, index_label = "gene_symbol")


Running: alpha vs other


In [19]:
# Loop over all cell types -----------------------------------------------------------------------------
for cluster_id in target_celltypes:

    meta_results = []

    print(f"\n==============================")
    print(f"Running: {cluster_id} vs other")
    print(f"==============================")

    comp = f"{cluster_id}_vs_other"
    
    adata.obs[comp] = (
        adata
        .obs[anno_key]
        .apply(lambda x: cluster_id if x == cluster_id else 'other'))
    
    adata.obs['assay'] = 'my_assay'

    for dataset in adata.obs[dataset_key].unique():

        # Subset per dataset
        ad_sub = adata[
            adata.obs[dataset_key] == dataset
        ].copy()

        # Skip studies with less than 100 cells
        if ad_sub.n_obs < 100:
            continue

           
        # Pseudobulk aggregation (by comparison group + sample)
        pb = dp.aggregate_pseudobulk(
            ad_sub,
            layer='counts',
            groupby=['assay', donor_key, comp]
        )

    
        try:
            min_cells = 50
            pb = dp.filter_samples(pb, min_cells=min_cells, min_samples=3)

            # If there are too few replicates, reduce minimum number of cells 
            if pb.obs.groupby(comp)['assay'].count()[cluster_id] < 3:
                pb = dp.aggregate_pseudobulk(
                ad_sub,
                layer='counts',
                groupby=['assay', donor_key, comp])

                min_cells = 10
                pb = dp.filter_samples(pb, min_cells=min_cells, min_samples=3)

            # Count matrix
            counts_df = pd.DataFrame(
                pb.X.toarray(),
                columns=pb.var_names,
                index=pb.obs_names)
            
            metadata_df = pb.obs[[donor_key, comp]].copy()
            metadata_df = metadata_df.set_index(counts_df.index)
            
            assert counts_df.index.equals(metadata_df.index), "Index are not equal!"

            # DDS analysis
            formula = "~ {} + {}".format(donor_key, comp)
            
            dds = DeseqDataSet(
                counts=counts_df,
                metadata=metadata_df,
                design= formula,
                inference=inference
            )
            
            dds.deseq2()
            
            ds = DeseqStats(
                dds,
                contrast=(comp, cluster_id, 'other'), 
                inference = inference,
                quiet = True)
            
            # run wald test
            ds.run_wald_test()
            ds.summary()
            

            # Calculate % of genes expressed in cell types (from cell in donors used for DE)
            perc = dg.compute_pct_expressing(ad_sub[ad_sub.obs[donor_key].isin(pb.obs[donor_key].reset_index(drop=True).drop_duplicates().to_list())],
                                             anno_key)
    
            # Count number of donors used for DE
            n_donor=pd.concat([pb.obs[disease_key].value_counts().to_frame().transpose(), pb.obs[comp].value_counts().to_frame().transpose()], axis = 1)
            perc[n_donor.columns.to_list()] = n_donor.iloc[0]
            perc
        
            
            results = ds.results_df.copy()
            n_donors = ds.dds.shape[0]
            results['comparison'] = f"{cluster_id}_other"
            results['manual_annotation'] = cluster_id
            results['n_donors'] = n_donors
            results['dataset'] = dataset
            results['min_cells'] = min_cells
    
            # combine with information
            results = results.join(perc, how = 'left')
            meta_results.append(results)
            
            results.to_csv(os.path.join(tmp_dir, f"{cluster_id}_other_{dataset}.csv"), index=True, index_label = "gene_symbol")
            
        except Exception as e:
            print("Skipping:", cluster_id, e)

        # Combine all results -------------------------------------------------------------------------
        # Only concatenate if there are results
        if len(meta_results) > 0:
            meta_df = pd.concat(meta_results)
            meta_df.to_csv(os.path.join(diffg_dir, f"deseq2_one_vs_all_per_dataset_{cluster_id}.csv"), index=True, index_label = "gene_symbol")
        else:
            print(f"Skipping entire cluster {cluster_id} - all datasets failed")
            continue  # Skip to next cluster


Running: stellate_quiescent vs other


/work/islet_cartography_scrna/scrna_cartography_deseq/lib/python3.13/functools.py:934: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
Fitting size factors...
... done in 0.05 seconds.



filter_samples: 39 samples dropped (n_cells < 50)
  my_assay: 45 samples retained
filter_samples: 45/84 samples, 1 assays retained, 0 dropped
Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 2.15 seconds.

Fitting dispersion trend curve...
/work/islet_cartography_scrna/scrna_cartography_deseq/lib/python3.13/site-packages/pydeseq2/dds.py:822: UserWarning: The dispersion trend curve fitting did not converge. Switching to a mean-based dispersion trend.
  self._fit_parametric_dispersion_trend(vst)
... done in 0.59 seconds.

/work/islet_cartography_scrna/scrna_cartography_deseq/lib/python3.13/site-packages/pydeseq2/dds.py:548: UserWarning: As the residual degrees of freedom is less than 3, the distribution of log dispersions is especially asymmetric and likely to be poorly estimated by the MAD.
  self.fit_dispersion_prior()
Fitting MAP dispersions...
... done in 2.13 seconds.

Fitting LFCs...
... done in 2.87 seconds.

Calculating cook's distance...
... done in 0.05 seconds.

Replacing 0 outlier genes.

/work/islet_cartography_scrna/scrna_cartography_deseq/lib/python3.13/functools.py:934: ImplicitModificationWarning: Transforming to str index.

filter_samples: 21 samples dropped (n_cells < 50)
  my_assay: 21 samples retained
filter_samples: 21/42 samples, 1 assays retained, 0 dropped
Skipping: stellate_quiescent 'stellate_quiescent'


...

/work/islet_cartography_scrna/scrna_cartography_deseq/lib/python3.13/functools.py:934: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


filter_samples: 2 samples dropped (n_cells < 50)
  my_assay: 4 samples retained
filter_samples: 4/6 samples, 1 assays retained, 0 dropped


/work/islet_cartography_scrna/scrna_cartography_deseq/lib/python3.13/functools.py:934: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
Fitting size factors...
... done in 0.01 seconds.



filter_samples: 1 samples dropped (n_cells < 10)
  my_assay: 5 samples retained
filter_samples: 5/6 samples, 1 assays retained, 0 dropped
Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 1.26 seconds.

Fitting dispersion trend curve...
/work/islet_cartography_scrna/scrna_cartography_deseq/lib/python3.13/site-packages/pydeseq2/dds.py:822: UserWarning: The dispersion trend curve fitting did not converge. Switching to a mean-based dispersion trend.
  self._fit_parametric_dispersion_trend(vst)
... done in 0.49 seconds.

/work/islet_cartography_scrna/scrna_cartography_deseq/lib/python3.13/site-packages/pydeseq2/dds.py:548: UserWarning: As the residual degrees of freedom is less than 3, the distribution of log dispersions is especially asymmetric and likely to be poorly estimated by the MAD.
  self.fit_dispersion_prior()
Fitting MAP dispersions...
... done in 1.61 seconds.

Fitting LFCs...
... done in 1.87 seconds.

Calculating cook's distance...
... done in 0.01 seconds.

Replacing 0 outlier genes.



In [15]:
meta_results

[]